# Unit 17 — Web Scraping and APIs for Business Data

A practical notebook for external data acquisition, HTML parsing, cleaning and API concepts.

## Learning Objectives
- Explain web scraping and APIs.
- Understand practical HTTP and HTML concepts.
- Extract structured information from HTML.
- Clean scraped values for analysis.
- Understand endpoints, parameters and JSON API responses.
- Discuss responsible and ethical data collection.

## Prerequisites
- Python basics
- Pandas and JSON
- File handling
- Basic EDA

## Class Delivery
Approx. 60 minutes. Use local reproducible examples first. Live web access is an optional extension.

> **Teaching pattern:** explain the concept → demonstrate → interpret → practise → connect to a business decision.

## 1. Why External Data Matters

Internal business systems provide only part of the picture.

A business may also use:

- public websites
- competitor pages
- online catalogues
- APIs
- open-data sources
- public reports
- platform analytics

The general pipeline is:

```text
External Source
      ↓
Collection
      ↓
Parsing
      ↓
Validation / Cleaning
      ↓
Pandas DataFrame
      ↓
Analysis
      ↓
Business Insight
```

In [ ]:
import pandas as pd
import numpy as np
from io import StringIO
print("Environment ready")

Environment ready


## 2. Web Scraping vs API

### Web scraping

Extract information from a webpage.

### API

Use a defined programmatic interface to request data.

| Web Scraping | API |
|---|---|
| Usually extracts HTML | Usually returns structured data |
| Depends on page structure | Depends on documented API contract |
| Can break when design changes | Usually more stable |
| Useful when no suitable API exists | Preferred when an official API exists |
| Must respect site restrictions | Must respect API terms and limits |

## 3. HTTP Concepts

You only need the practical concepts:

- URL
- request
- response
- status code
- headers
- query parameters
- content type

Common response codes:

| Code | Meaning |
|---:|---|
| 200 | Success |
| 301/302 | Redirect |
| 400 | Bad request |
| 401 | Authentication required |
| 403 | Forbidden |
| 404 | Not found |
| 429 | Too many requests |
| 500 | Server-side error |

## 4. HTML Basics

Example:

```html
<div class="product">
    <h2 class="name">Wireless Mouse</h2>
    <span class="price">900</span>
</div>
```

Important ideas:

- tag/element
- attribute
- class
- id
- nested elements
- text content

The goal is not to teach web development. It is to give students enough understanding to locate data.

## 5. Create Local HTML for Reproducible Teaching

The notebook uses local HTML examples so that every student can run the lesson without depending on a live website.

In [ ]:
html = '''
<html>
  <body>
    <div class="product">
      <h2 class="name">Wireless Mouse</h2>
      <span class="price">900</span>
      <span class="rating">4.4</span>
    </div>
    <div class="product">
      <h2 class="name">Mechanical Keyboard</h2>
      <span class="price">1800</span>
      <span class="rating">4.6</span>
    </div>
    <div class="product">
      <h2 class="name">USB Webcam</h2>
      <span class="price">2500</span>
      <span class="rating">4.2</span>
    </div>
  </body>
</html>
'''
print(html[:250], "...")


<html>
  <body>
    <div class="product">
      <h2 class="name">Wireless Mouse</h2>
      <span class="price">900</span>
      <span class="rating">4.4</span>
    </div>
    <div class="product">
      <h2 class="name">Mechanical Keyboard</h2>
     ...


## 6. Parse HTML with BeautifulSoup

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")
print(soup.get_text(" ", strip=True))

Wireless Mouse 900 4.4 Mechanical Keyboard 1800 4.6 USB Webcam 2500 4.2


### 6.1 Select One Element

In [ ]:
first_name = soup.select_one(".name").get_text(strip=True)
first_price = soup.select_one(".price").get_text(strip=True)

print("Name:", first_name)
print("Price:", first_price)

Name: Wireless Mouse
Price: 900


### 6.2 Select Multiple Elements

In [ ]:
names = [node.get_text(strip=True) for node in soup.select(".name")]
prices = [node.get_text(strip=True) for node in soup.select(".price")]
ratings = [node.get_text(strip=True) for node in soup.select(".rating")]

print(names)
print(prices)
print(ratings)

['Wireless Mouse', 'Mechanical Keyboard', 'USB Webcam']
['900', '1800', '2500']
['4.4', '4.6', '4.2']


### 6.3 HTML → DataFrame

In [ ]:
products = pd.DataFrame({
    "product": names,
    "price": pd.to_numeric(prices),
    "rating": pd.to_numeric(ratings)
})
products

,product,price,rating
0,Wireless Mouse,900,4.4
1,Mechanical Keyboard,1800,4.6
2,USB Webcam,2500,4.2


## 7. Finding Elements More Precisely

Useful selectors:

```text
.product
.name
#main
div.product
div.product .price
```

The exact selector depends on the HTML structure.

### Teaching point

A scraper is tightly coupled to the structure of the source. If the website changes its HTML, the code may stop working.

In [ ]:
for product in soup.select(".product"):
    name = product.select_one(".name").get_text(strip=True)
    price = product.select_one(".price").get_text(strip=True)
    print(name, "→", price)

Wireless Mouse → 900
Mechanical Keyboard → 1800
USB Webcam → 2500


## 8. Extracting HTML Tables

Business websites often publish tables.

Pandas can read tables from HTML when the page exposes a standard table structure.

In [ ]:
table_html = '''
<table id="sales">
<tr><th>Product</th><th>Region</th><th>Revenue</th></tr>
<tr><td>Mouse</td><td>North</td><td>90000</td></tr>
<tr><td>Keyboard</td><td>North</td><td>150000</td></tr>
<tr><td>Webcam</td><td>South</td><td>120000</td></tr>
<tr><td>Monitor</td><td>South</td><td>220000</td></tr>
</table>
'''

tables = pd.read_html(StringIO(table_html))
tables[0]

,Product,Region,Revenue
0,Mouse,North,90000
1,Keyboard,North,150000
2,Webcam,South,120000
3,Monitor,South,220000


## 9. `requests` — Getting Web Content

Typical pattern:

```python
import requests

response = requests.get(url, timeout=10)
response.raise_for_status()
html = response.text
```

However, live websites can:

- block bots
- require authentication
- rate-limit requests
- change their HTML
- return dynamic content
- impose legal or contractual restrictions

Therefore, this notebook keeps the main class exercises local and reproducible.

In [ ]:
# Optional live example — only use a website you are permitted to access.
#
# import requests
# url = "https://example.com"
# response = requests.get(url, timeout=10)
# response.raise_for_status()
# print(response.status_code)
# print(response.text[:500])

## 10. Cleaning Scraped Data

Scraped values often arrive as strings:

- `₹1,25,000`
- `"  Mouse "`
- `"4.6/5"`
- `"10%"`
- `"N/A"`

A data-acquisition workflow must therefore include cleaning and validation.

In [ ]:
scraped = pd.DataFrame({
    "product": [" Mouse ", "Keyboard", "Webcam", "Monitor"],
    "price": ["₹900", "₹1,800", "₹2,500", "₹12,500"],
    "rating": ["4.4/5", "4.6/5", None, "4.8/5"]
})

scraped["product"] = scraped["product"].str.strip()
scraped["price"] = (
    scraped["price"]
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)
scraped["rating"] = (
    scraped["rating"]
    .str.replace("/5", "", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

scraped

,product,price,rating
0,Mouse,900.0,4.4
1,Keyboard,1800.0,4.6
2,Webcam,2500.0,NaN
3,Monitor,12500.0,4.8


## 11. Business Example — Competitor Price Monitoring

Business question:

> How do our prices compare with competitors?

Possible pipeline:

```text
Competitor Websites
       ↓
Scrape Product + Price
       ↓
Standardise Names
       ↓
Validate Data
       ↓
Compare Prices
       ↓
Visualise
       ↓
Business Recommendation
```

In [ ]:
competitor_prices = pd.DataFrame({
    "product": ["Mouse", "Keyboard", "Webcam", "Monitor"],
    "our_price": [900, 1800, 2500, 12500],
    "competitor_a": [850, 1950, 2400, 13000],
    "competitor_b": [920, 1750, 2600, 12200]
})

competitor_prices["competitor_avg"] = competitor_prices[
    ["competitor_a", "competitor_b"]
].mean(axis=1)

competitor_prices["price_gap"] = (
    competitor_prices["our_price"] - competitor_prices["competitor_avg"]
)

competitor_prices

,product,our_price,competitor_a,competitor_b,competitor_avg,price_gap
0,Mouse,900,850,920,885.0,15.0
1,Keyboard,1800,1950,1750,1850.0,-50.0
2,Webcam,2500,2400,2600,2500.0,0.0
3,Monitor,12500,13000,12200,12600.0,-100.0


In [ ]:
plt.figure(figsize=(10, 5))
x = np.arange(len(competitor_prices))
width = 0.25

plt.bar(x - width, competitor_prices["our_price"], width, label="Our Price")
plt.bar(x, competitor_prices["competitor_a"], width, label="Competitor A")
plt.bar(x + width, competitor_prices["competitor_b"], width, label="Competitor B")

plt.xticks(x, competitor_prices["product"])
plt.ylabel("Price")
plt.title("Competitor Price Comparison")
plt.legend()
plt.show()

NameError: name 'plt' is not defined

## 12. API Fundamentals

An API request commonly involves:

- endpoint
- method
- parameters
- headers
- authentication
- response

A common response format is JSON.

Example:

```json
{
  "status": "ok",
  "data": [
      {"city": "Delhi", "temperature": 31}
  ]
}
```

## 13. Use a Mock API Response

A mock response makes the lesson reproducible and keeps students focused on the data workflow rather than API keys.

In [ ]:
mock_api_response = {
    "status": "ok",
    "data": [
        {"city": "Delhi", "temperature": 31, "humidity": 55},
        {"city": "Mumbai", "temperature": 29, "humidity": 72},
        {"city": "Chennai", "temperature": 33, "humidity": 68}
    ]
}

mock_api_response

In [ ]:
api_df = pd.DataFrame(mock_api_response["data"])
api_df

## 14. Query Parameters

A request may include parameters such as:

```text
city=Delhi
units=metric
limit=10
```

In Python:

```python
params = {"city": "Delhi", "units": "metric"}
requests.get(url, params=params)
```

Parameters make requests easier to construct and debug.

In [ ]:
params = {"city": "Delhi", "units": "metric", "limit": 10}
params

## 15. API Error Handling

A robust collection workflow should consider:

- timeout
- connection error
- non-success status
- malformed JSON
- missing fields
- rate limits
- retry strategy

Students should understand the design principle even when advanced API engineering is taught elsewhere.

In [ ]:
def parse_api_response(payload):
    if payload.get("status") != "ok":
        raise ValueError("API returned a non-success status")
    if "data" not in payload:
        raise KeyError("Expected 'data' field is missing")
    return pd.DataFrame(payload["data"])

parse_api_response(mock_api_response)

## 16. Responsible Web Data Collection

Discuss:

- Terms of service
- robots.txt
- rate limits
- personal data
- authentication
- copyright/content ownership
- data minimisation
- official APIs where available
- avoiding unnecessary load on a server

The lesson is not “scrape everything”; it is **collect data responsibly and for a legitimate business purpose**.

## 17. Guided Class Exercise — Competitor Monitoring

Students must:

1. Create a local HTML page containing five products.
2. Scrape product names and prices.
3. Clean the values.
4. Convert them to numeric types.
5. Compare them to a second competitor dataset.
6. Calculate price gaps.
7. Create one visualisation.
8. Write two evidence-based recommendations.

In [ ]:
# Student starter workspace
exercise_html = '''
<div class="item"><span class="name">Product A</span><span class="price">₹1000</span></div>
<div class="item"><span class="name">Product B</span><span class="price">₹1500</span></div>
<div class="item"><span class="name">Product C</span><span class="price">₹2200</span></div>
<div class="item"><span class="name">Product D</span><span class="price">₹3500</span></div>
<div class="item"><span class="name">Product E</span><span class="price">₹4800</span></div>
'''

# Write your own BeautifulSoup extraction code below.

## 18. Common Mistakes

- Hard-coding assumptions about HTML structure.
- Failing to clean extracted strings.
- Ignoring missing values.
- Sending repeated requests too quickly.
- Ignoring website/API restrictions.
- Treating scraped data as automatically correct.
- Not recording when and where the data was collected.

## 19. Summary

Students should now understand:

- how data can be acquired from external websites and APIs
- basic HTML structure and selectors
- how BeautifulSoup can extract information
- how scraped values become analysis-ready data
- the basic API request/response model
- how JSON becomes a DataFrame
- the importance of responsible collection

Next: **Predictive Analysis and Forecasting — estimating what may happen in the future without introducing machine-learning models.**